# 🎓 Student Success Classification Model
**Descripción:** Guía completa de referencia y aprendizaje sobre el proyecto "Predict students' dropout and academic success".
**Autor:** [Tu Nombre]
**Fecha:** [Fecha]
**Tecnologías:** Python, SQL, scikit-learn, Power BI


## 1. Definición del Problema
**Contexto de Negocio:** Las instituciones educativas necesitan identificar proactivamente a los estudiantes en riesgo de abandonar sus estudios a principios de semestre.

**Las 3 Clases Objetivo:**
- **Dropout (Abandono):** El estudiante abandonó sus estudios. (Grupo crítico).
- **Enrolled (Matriculado):** El estudiante sigue activo, pero podría estar en riesgo.
- **Graduate (Graduado):** El estudiante completó el programa con éxito.

**Propuesta de Valor:**
Predecir de forma temprana el "Dropout" permite a la institución intervenir con becas, tutorías académicas o apoyo psicológico antes de que el estudiante tome la decisión de irse.

**Variables del Dataset:**
- Demográficas: Edad al matricularse, género, nacionalidad.
- Socioeconómicas: Becas, estado de las cuotas de matrícula.
- Métricas Académicas: Unidades curriculares aprobadas, calificaciones del 1º y 2º semestre.


## 2. Arquitectura del Repositorio
Así es como organizamos el código del proyecto siguiendo las mejores prácticas de ingeniería de software. Cada archivo tiene una única responsabilidad.

```text
Student-Success-Classification-Model/
├── data/           (raw/, interim/, processed/)
├── notebooks/      (01_eda.ipynb)
├── sql/            (exploration.sql)
├── src/            (Código fuente Python)
├── config/         (config.yaml)
├── models/         (Modelos entrenados guardados)
├── reports/        (Resultados en CSV y métricas)
├── requirements.txt
└── README.md
```

| Archivo | Responsabilidad Única |
|---|---|
| `load_data.py` | Descargar el dataset original y guardarlo. |
| `sql_exploration.py` | Cargar el CSV en SQLite y ejecutar consultas SQL. |
| `preprocess.py` | Limpiar los datos (Nulos, valores inválidos). |
| `feature_engineering.py` | Transformar variables y seleccionar las más relevantes. |
| `train_model.py` | Entrenar el algoritmo de clasificación (Random Forest). |
| `evaluate_model.py` | Evaluar el modelo y exportar las métricas de rendimiento. |
| `predict.py` | Generar nuevas predicciones con datos no vistos. |
| `exploration.sql` | Almacenar las consultas de exploración SQL documentadas. |


## 3. Configuración del Proyecto
**¿Por qué usamos config.yaml en lugar de valores "hardcodeados"?**
Escribir rutas o parámetros directamente en el código Python (ej. `df.to_csv('data/raw/data.csv')`) es una mala práctica porque hace que el código sea frágil y difícil de mantener. Al centralizar esto en un archivo YAML, cualquier persona (incluso sin saber programar) puede ajustar el modelo o las rutas desde un solo lugar.


In [ ]:
# config.yaml (Representado aquí como variable para referencia rápida si quieres cargarlo, aunque en el repositorio es un archivo físico)

config_yaml_content = """
data_source:
  uci_dataset_id: 697

paths:
  raw: "data/raw/"
  interim: "data/interim/"
  processed: "data/processed/"
  database: "data/interim/student_database.sqlite"
  models: "models/"
  model_filename: "student_rf_model.joblib"

training:
  target_column_original: "Target"
  target_column: "Target_Encoded"
  test_size: 0.2
  random_state: 42

feature_engineering:
  selected_features:
    - "Marital_status"
    - "Application_mode"
    - "Application_order"
    - "Course"
    - "Previous_qualification"
    - "Nacionality"
    - "Scholarship_holder"
    - "Age_at_enrollment"
    - "Curricular_units_1st_sem_approved"
    - "Curricular_units_1st_sem_grade"
    - "Curricular_units_2nd_sem_approved"
    - "Curricular_units_2nd_sem_grade"

  target_mapping:
    Graduate: 0
    Dropout: 1
    Enrolled: 2

model:
  n_estimators: 100
  max_depth: 10
  random_state: 42

evaluation:
  class_names: ["Graduate", "Dropout", "Enrolled"]
"""


## 4. Resumen del Pipeline ML + SQL
El flujo de trabajo se divide en 7 pasos lineales y secuenciales:

| Paso | Script | Propósito |
|---|---|---|
| **1** | `load_data.py` | Descargar datos crudos desde UCI. |
| **2** | `sql_exploration.py` | Inspeccionar los datos con consultas SQL. |
| **3** | `preprocess.py` | Limpiar valores nulos y anómalos. |
| **4** | `feature_engineering.py` | Crear características útiles para el modelo. |
| **5** | `train_model.py` | Entrenar el Random Forest Classifier. |
| **6** | `evaluate_model.py` | Generar métricas de precisión. |
| **7** | `predict.py` | Inferir riesgo en nuevos alumnos. |


## 5. Paso 1 — Carga de Datos
El primer paso es automatizar la obtención de los datos. Esto asegura que si el dataset original se actualiza, podemos descargar la nueva versión con un solo comando en lugar de bajar el CSV manualmente desde nuestro navegador.


In [ ]:
import pandas as pd
import yaml
from ucimlrepo import fetch_ucirepo

def load_config(config_path="config/config.yaml"):
    with open(config_path, "r") as file:
        return yaml.safe_load(file)

def fetch_and_save_data():
    config = load_config()
    dataset_id = config['data_source']['uci_dataset_id']
    raw_path = config['paths']['raw'] + "data.csv"
    
    print(f"Fetching dataset with ID {dataset_id} from UCI...")
    dataset = fetch_ucirepo(id=dataset_id)
    
    df = dataset.data.features.copy()
    df['Target'] = dataset.data.targets
    
    df.to_csv(raw_path, index=False)
    print(f"Raw data successfully saved to: {raw_path}")


## 6. Paso 2 — Exploración SQL
**¿Por qué usamos SQL antes de Python?**
El lenguaje SQL es el estándar de la industria para recuperar información de bases de datos. Usar una base de datos local como `SQLite` dentro de nuestro pipeline de Python demuestra que sabes integrar ambos mundos: puedes traer datos, transformarlos y luego hacerles el perfilado (profiling) inicial usando la herramienta más demandada en análisis de datos.

A continuación, el script de Python que carga la base de datos y ejecuta las consultas.


In [ ]:
import sqlite3
import pandas as pd
import yaml
import os

def execute_sql_exploration():
    config = yaml.safe_load(open("config/config.yaml", "r"))
    raw_path = config['paths']['raw'] + "data.csv"
    db_path = config['paths']['database']
    sql_script_path = "sql/exploration.sql"
    reports_dir = "reports/"
    os.makedirs(reports_dir, exist_ok=True)
    
    df = pd.read_csv(raw_path)
    conn = sqlite3.connect(db_path)
    df.to_sql('student_data', conn, if_exists='replace', index=False)
    
    with open(sql_script_path, "r") as file:
        sql_queries = file.read().split(';')
    
    query_names = ["inspection", "target_distribution", "scholarship_analysis", "data_quality"]
    
    for i, query in enumerate(sql_queries):
        query = query.strip()
        if not query: continue
        result_df = pd.read_sql_query(query, conn)
        output_csv = f"{reports_dir}sql_{query_names[i]}.csv"
        result_df.to_csv(output_csv, index=False)
    
    conn.close()


### Y este es el archivo SQL (`exploration.sql`)
Cada consulta tiene un objetivo específico:
1. **Inspection**: Verifica que 1 fila = 1 estudiante.
2. **Target Distribution**: Busca desbalanceo de clases (si casi todos son Graduados, el modelo tendrá problemas para aprender sobre los Abandonos).
3. **Scholarship Analysis**: Comprueba si tener una beca reduce drásticamente el abandono.
4. **Data Quality Check**: Detecta valores anómalos (ej. edades imposibles) que deberán ser removidos en el paso 3.


In [ ]:
-- 1. Dataset Inspection
SELECT *
FROM student_data
LIMIT 10;

-- 2. Target Variable Distribution
SELECT 
    Target, 
    COUNT(*) AS student_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM student_data
GROUP BY Target
ORDER BY student_count DESC;

-- 3. Feature vs Target Aggregation (Scholarship Impact)
SELECT 
    Scholarship_holder,
    Target,
    COUNT(*) AS student_count
FROM student_data
GROUP BY Scholarship_holder, Target
ORDER BY Scholarship_holder, Target;

-- 4. Data Quality Check
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Age_at_enrollment IS NULL THEN 1 ELSE 0 END) AS missing_age,
    SUM(CASE WHEN Age_at_enrollment < 16 OR Age_at_enrollment > 80 THEN 1 ELSE 0 END) AS invalid_age
FROM student_data;


## 7. Paso 3 — Preprocesamiento de Datos
**¿Qué significa limpiar los datos?**
Los datos en la vida real siempre están sucios. Tienen valores faltantes (NULLs) o absurdos (una persona con 150 años). Si le damos datos basura a nuestro modelo de Machine Learning, aprenderá basura (Garbage In, Garbage Out). Este script remueve a cualquier estudiante sin objetivo definido y sanea las edades imposibles detectadas en la fase SQL.


In [ ]:
import pandas as pd
import yaml

def clean_data():
    config = yaml.safe_load(open("config/config.yaml", "r"))
    input_path = config['paths']['raw'] + "data.csv"
    output_path = config['paths']['interim'] + "cleaned_data.csv"
    
    df = pd.read_csv(input_path)
    target_col = config['training']['target_column_original']
    df = df.dropna(subset=[target_col])
    
    df = df[(df['Age_at_enrollment'] >= 16) & (df['Age_at_enrollment'] <= 80)]
    
    df.to_csv(output_path, index=False)


## 8. Paso 4 — Ingeniería de Características (Feature Engineering)
**¿Qué es Feature Engineering y por qué lo usamos?**
Los modelos matemáticos (como Random Forest) solo entienden números, no texto. Aquí convertimos nuestras clases ("Graduate", "Dropout") en números (0, 1, 2). También aprovechamos para descartar las columnas de ruido que no aportan valor predictivo, quedándonos solo con las que configuramos como relevantes.


In [ ]:
import pandas as pd
import yaml

def build_features():
    config = yaml.safe_load(open("config/config.yaml", "r"))
    input_path = config['paths']['interim'] + "cleaned_data.csv"
    output_path = config['paths']['processed'] + "engineered_data.csv"
    
    df = pd.read_csv(input_path)
    
    target_mapping = config['feature_engineering']['target_mapping']
    original_target = config['training']['target_column_original']
    encoded_target = config['training']['target_column']
    
    df[encoded_target] = df[original_target].map(target_mapping)
    
    selected_features = config['feature_engineering']['selected_features']
    columns_to_keep = selected_features + [encoded_target]
    df = df[columns_to_keep]
    
    df.to_csv(output_path, index=False)


## 9. Paso 5 — Entrenamiento del Modelo
**¿Qué es un Random Forest?**
Imagina que consultas a un solo experto para saber si un alumno va a abandonar. Si solo le preguntas a una persona, podría equivocarse (esto se llama un Árbol de Decisión). Random Forest le pregunta a un "bosque" de 100 expertos diferentes al mismo tiempo y toma la respuesta que decida la mayoría. Es poderoso y resistente a errores.

**¿Por qué separamos datos en Entrenamiento y Prueba (Train/Test Split)?**
Es como darle un examen a un estudiante. Si le das las mismas preguntas con las que estudió (entrenamiento), sacará un 100%, pero eso no demuestra que haya aprendido. Para evaluar verdaderamente al modelo, debemos probarlo usando un 20% de datos (Prueba) que **jamás ha visto antes**.


In [ ]:
import pandas as pd
import yaml
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

def train_classification_model():
    config = yaml.safe_load(open("config/config.yaml", "r"))
    input_path = config['paths']['processed'] + "engineered_data.csv"
    model_path = config['paths']['models'] + config['paths']['model_filename']
    
    df = pd.read_csv(input_path)
    target_col = config['training']['target_column']
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config['training']['test_size'],
        random_state=config['training']['random_state']
    )
    
    model = RandomForestClassifier(
        n_estimators=config['model']['n_estimators'],
        max_depth=config['model']['max_depth'],
        random_state=config['model']['random_state']
    )
    model.fit(X_train, y_train)
    
    joblib.dump(model, model_path)
    X_test.to_csv(config['paths']['interim'] + "X_test.csv", index=False)
    y_test.to_csv(config['paths']['interim'] + "y_test.csv", index=False)


## 10. Paso 6 — Evaluación del Modelo
**Diccionario de Métricas:**
- **Precisión (Precision):** De todos los que predijimos como "Dropout", ¿cuántos lo fueron realmente? (Evitamos falsas alarmas que malgasten a los tutores).
- **Recuerdo (Recall):** De todos los "Dropouts" reales que había, ¿qué porcentaje logró capturar el modelo? (Vital: No queremos que un alumno se nos escape).
- **F1-Score:** El equilibrio perfecto entre Precisión y Recuerdo.

**Visión de Negocio:** Siempre que tratamos de retener clientes (o alumnos), optimizamos el Recall en la clase de Abandono. Es mejor llamar a un alumno de más (falso positivo) y ofrecerle ayuda, a que dejar que uno abandone sin intervenir (falso negativo).


In [ ]:
import pandas as pd
import yaml
import joblib
from sklearn.metrics import classification_report

def evaluate_predictions():
    config = yaml.safe_load(open("config/config.yaml", "r"))
    model_path = config['paths']['models'] + config['paths']['model_filename']
    X_test_path = config['paths']['interim'] + "X_test.csv"
    y_test_path = config['paths']['interim'] + "y_test.csv"
    
    model = joblib.load(model_path)
    X_test = pd.read_csv(X_test_path)
    y_test = pd.read_csv(y_test_path)
    
    predictions = model.predict(X_test)
    class_names = config['evaluation']['class_names']
    
    report_text = classification_report(y_test, predictions, target_names=class_names)
    print(report_text)
    
    report_dict = classification_report(y_test, predictions, target_names=class_names, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv("reports/model_evaluation_metrics.csv")


## 11. Paso 7 — Generación de Predicciones
**¿Qué significa Inferencia (Inference)?**
A la hora de usar el modelo en el mundo real, tomamos los datos frescos del nuevo semestre (Inferencia), los pasamos por nuestro archivo entrenado `.joblib`, y este nos devuelve una calificación de riesgo ("Graduate", "Dropout", etc.).


In [ ]:
import pandas as pd
import yaml
import joblib

def make_new_predictions(input_csv):
    config = yaml.safe_load(open("config/config.yaml", "r"))
    model_path = config['paths']['models'] + config['paths']['model_filename']
    output_path = config['paths']['processed'] + "new_predictions.csv"
    
    model = joblib.load(model_path)
    df_new = pd.read_csv(input_csv)
    
    selected_features = config['feature_engineering']['selected_features']
    df_features = df_new[selected_features]
    
    predictions = model.predict(df_features)
    df_new['Predicted_Target_Code'] = predictions
    
    target_mapping = config['feature_engineering']['target_mapping']
    reverse_mapping = {value: key for key, value in target_mapping.items()}
    df_new['Risk_Category'] = df_new['Predicted_Target_Code'].map(reverse_mapping)
    
    df_new.to_csv(output_path, index=False)


## 12. Guía de Ejecución
Para que este proyecto funcione desde cero, debes seguir un orden estricto (no puedes predecir sin antes haber descargado datos y entrenado el modelo).


In [ ]:
# Comando de instalación inicial
!pip install -r requirements.txt

# Recomendamos correr esto desde la terminal de tu IDE (ej. Bash o PowerShell)
# python src/load_data.py           # Paso 1: Descarga los datos crudos
# python src/sql_exploration.py     # Paso 2: Saca reportes con SQLite
# python src/preprocess.py          # Paso 3: Limpia datos nulos/inválidos
# python src/feature_engineering.py # Paso 4: Construye variables maestras
# python src/train_model.py         # Paso 5: Crea y guarda el experto virtual
# python src/evaluate_model.py      # Paso 6: Exporta el boletín de notas del experto
# python src/predict.py             # Paso 7: Genera las predicciones para producción


## 13. Flujo de Trabajo en Git (Git Workflow)
**Tip para Juniors:** Usar "Conventional Commits" (`feat`, `docs`, `fix`) demuestra madurez e higiene en el código. Esto comunica a un reclutador que sabes trabajar en estándares abiertos.


In [ ]:
# git checkout -b develop
# git add .
# git commit -m "build: setup project architecture and requirements"
# git commit -m "feat: implement load_data script for UCI API"
# git commit -m "feat: add SQL exploration script and query file"
# git commit -m "feat: add data cleaning logic in preprocess.py"
# git commit -m "feat: implement feature engineering and target encoding"
# git commit -m "feat: add random forest model training script"
# git commit -m "feat: add evaluation reporting script"
# git commit -m "feat: implement predicting logic for new student records"
# git commit -m "docs: create README and PLAYBOOK guides"
# git commit -m "docs: add PowerBI documentation and final polish"
# git push origin develop


## 14. Guía para Dashboard en Power BI
Un modelo de ML gana su valor total cuando el equipo de negocio lee los resultados. Aquí conectamos nuestros `.csv` de `reports/` a Power BI.

**Fuentes a conectar:** `sql_target_distribution.csv`, `sql_scholarship_analysis.csv`, `model_evaluation_metrics.csv`

**Estructura del Tablero de 2 Páginas:**

### PÁGINA 1: Analítica Descriptiva (Lo que dicta SQL)
*Para dar contexto histórico.*

1. **Student Status Distribution** (Donut Chart)
   - *Campos*: Target, student_count
   - *Narrativa:* Muestra la proporción original de graduados vs abandono.
2. **Scholarship Impact on Dropouts** (100% Stacked Bar Chart)
   - *Campos*: Eje: Scholarship_holder. Leyenda: Target. Valores: student_count.
   - *Narrativa:* Prueba el peso de la ayuda financiera para prevenir el abandono.
3. **Data Quality Health** (Card / KPIs)
   - *Campos*: missing_age, invalid_age
   - *Narrativa:* Indicador para saber si los registros ingresados al modelo están limpios.

### PÁGINA 2: Analítica Predictiva (El Poder del Machine Learning)
*Para accionar de forma inmediata.*

4. **Model Accuracy Matrix** (Matrix visual)
   - *Campos*: precision, recall, f1-score
   - *Narrativa:* Demuestra que nuestras predicciones son de fiar.
5. **Risk Category Forecast** (Column Chart)
   - *Campos*: Risk_Category
   - *Narrativa:* Muestra cuántos alumnos actuales están catalogados en riesgo.
6. **Intervention Priority List** (Data Table)
   - *Campos*: ID de Estudiante + Predicted_Target_Code
   - *Narrativa:* Lista roja con los alumnos a contactar prioritariamente hoy.


## 15. Aprendizajes Clave (Key Learnings)
Has concluido con éxito este proyecto. ¿Qué le demuestras a una empresa?

1. **SQL + Python Híbrido:** Usaste SQLite integrado dentro de un pipeline de Machine Learning sin depender de servidores externos.
2. **Arquitectura Limpia:** Construiste todo con `config.yaml`, archivos Python delimitados y sin anidaciones complejas, superando el típico "jupyter sucio".
3. **Machine Learning Ético:** Optimizaste por Recall para que no quede ningún estudiante fuera del radar.
4. **Visión de BI y Negocio:** Entiendes que el código tiene un fin: tomar decisiones. Usar estos insights en Power BI cierra el ciclo de datos.

¡Enhorabuena!
